In [1]:
%load_ext autoreload
%autoreload 2
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import normflows as nf
import formula as fo
import summarize as su
import getplot as pl
import time
from torch.distributions import Normal

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [115]:
def _to_cpu(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu()
    return x


def to_tensor(x, dtype=torch.float32):
    if isinstance(x, torch.Tensor):
        return x.to(dtype)
    return torch.tensor(x, dtype=dtype)


def _safe_div(a, b, default=0.0):
    return a / b if b != 0 else default


# ----------------------------
# Small MLP used in invertible coupling layers
# ----------------------------
class CouplingMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_units=128, num_hidden_layers=2):
        super().__init__()
        layers = []
        last = in_dim
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(last, hidden_units))
            layers.append(nn.ReLU())
            last = hidden_units
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ----------------------------
# Affine coupling layer used as one invertible block of g_theta
# ----------------------------
class AffineCoupling(nn.Module):
    """
    Generic invertible affine coupling block.

    Here this layer is used as one component of the model-side
    transport map g_theta, not as the threshold mechanism itself.
    The threshold semantics are imposed later, after transport,
    in the (s, u, t) representation.
    """
    def __init__(self, dim, mask, hidden_units=128, num_hidden_layers=2, scale_clip=2.0):
        super().__init__()
        self.dim = dim
        self.register_buffer("mask", mask.view(1, dim))
        self.scale_clip = float(scale_clip)

        self.s_net = CouplingMLP(dim, dim, hidden_units, num_hidden_layers)
        self.t_net = CouplingMLP(dim, dim, hidden_units, num_hidden_layers)

        # near-identity initialization
        nn.init.zeros_(self.s_net.net[-1].weight)
        nn.init.zeros_(self.s_net.net[-1].bias)
        nn.init.zeros_(self.t_net.net[-1].weight)
        nn.init.zeros_(self.t_net.net[-1].bias)

    def forward(self, x, return_logdet=False):
        x_masked = x * self.mask
        raw_log_scale = self.s_net(x_masked) * (1.0 - self.mask)
        shift = self.t_net(x_masked) * (1.0 - self.mask)

        log_scale = self.scale_clip * torch.tanh(raw_log_scale / self.scale_clip)
        y = x_masked + (1.0 - self.mask) * (x * torch.exp(log_scale) + shift)

        if return_logdet:
            logdet = log_scale.sum(dim=-1)
            return y, logdet
        return y

    def inverse(self, y, return_logdet=False):
        y_masked = y * self.mask
        raw_log_scale = self.s_net(y_masked) * (1.0 - self.mask)
        shift = self.t_net(y_masked) * (1.0 - self.mask)

        log_scale = self.scale_clip * torch.tanh(raw_log_scale / self.scale_clip)
        x = y_masked + (1.0 - self.mask) * ((y - shift) * torch.exp(-log_scale))

        if return_logdet:
            logdet = (-log_scale).sum(dim=-1)
            return x, logdet
        return x


# ----------------------------
# Identity transport: basic formulation special case g_theta(eps)=eps
# ----------------------------
class IdentityTransport(nn.Module):
    """
    Special case corresponding to the basic latent-space formulation:
        g_theta(eps) = eps
    so no separate transport is learned.
    """
    def forward(self, eps, return_logdet=False):
        if return_logdet:
            return eps, eps.new_zeros(eps.shape[0])
        return eps

    def inverse(self, z, return_logdet=False):
        if return_logdet:
            return z, z.new_zeros(z.shape[0])
        return z


# ----------------------------
# Learnable invertible transport map g_theta
# ----------------------------
class TransportMap(nn.Module):
    """
    Model-side transport map g_theta in the advanced formulation.

    Input:
        eps in R^d, with d = 2p + 1
    Output:
        xi = g_theta(eps) in R^(2p+1)

    The block interpretation xi = (s, u, t) is imposed downstream
    by the target/decoder, not inside this generic transport class.
    """
    def __init__(self, dim, K=4, hidden_units=128, num_hidden_layers=2):
        super().__init__()
        self.dim = dim
        layers = []

        base_mask = torch.tensor(
            [1 if i % 2 == 0 else 0 for i in range(dim)],
            dtype=torch.float32
        )

        for k in range(K):
            mask = base_mask if (k % 2 == 0) else (1.0 - base_mask)
            layers.append(
                AffineCoupling(
                    dim=dim,
                    mask=mask,
                    hidden_units=hidden_units,
                    num_hidden_layers=num_hidden_layers,
                )
            )

        self.layers = nn.ModuleList(layers)

    def forward(self, eps, return_logdet=False):
        z = eps
        if return_logdet:
            total = eps.new_zeros(eps.shape[0])
        for layer in self.layers:
            if return_logdet:
                z, ld = layer(z, return_logdet=True)
                total = total + ld
            else:
                z = layer(z)
        if return_logdet:
            return z, total
        return z

    def inverse(self, z, return_logdet=False):
        x = z
        if return_logdet:
            total = z.new_zeros(z.shape[0])
        for layer in reversed(self.layers):
            if return_logdet:
                x, ld = layer.inverse(x, return_logdet=True)
                total = total + ld
            else:
                x = layer.inverse(x)
        if return_logdet:
            return x, total
        return x

In [116]:
class Relaxedsas(nn.Module):
    def __init__(
        self,
        X,
        y,
        sigma2,
        tau=0.5,
        slab_scale=1.0,
        mu_t=0.0,
        sigma_t=1.0,
        eps=1e-8,
        g_theta=None,
    ):
        super().__init__()
        self.register_buffer("X", X)
        self.register_buffer("y", y)
        self.register_buffer("sigma2", torch.as_tensor(float(sigma2), dtype=X.dtype, device=X.device))
        self.register_buffer("tau", torch.as_tensor(float(tau), dtype=X.dtype, device=X.device))
        self.register_buffer("slab_scale", torch.as_tensor(float(slab_scale), dtype=X.dtype, device=X.device))
        self.register_buffer("mu_t", torch.as_tensor(float(mu_t), dtype=X.dtype, device=X.device))
        self.register_buffer("sigma_t", torch.as_tensor(float(sigma_t), dtype=X.dtype, device=X.device))

        self.eps = eps
        self.n, self.p = X.shape

        self.dim = 2 * self.p + 1
        self.g_theta = g_theta if g_theta is not None else IdentityTransport()

    def set_tau(self, tau):
        self.tau.fill_(float(tau))

    def split_latent(self, z):
        p = self.p
        s = z[:, 0:p]
        u = z[:, p:2*p]
        t = z[:, 2*p:2*p+1]
        return s, u, t

    def decode(self, z):
        epsilon = z
        xi = self.g_theta(epsilon)

        s, u, t = self.split_latent(xi)

        m = torch.sigmoid((u - t) / self.tau)
        beta = s * m

        return {
            "epsilon": epsilon,
            "xi": xi,
            "s": s,
            "u": u,
            "t": t,
            "m": m,
            "beta": beta,
        }

    def log_prob(self, z):
        dec = self.decode(z)
        epsilon = dec["epsilon"]
        beta = dec["beta"]

        n = self.n

        fit = self.X @ beta.T
        resid = self.y[:, None] - fit
        loglik = -0.5 * (
            (resid ** 2).sum(dim=0) / self.sigma2
            + n * torch.log(2.0 * torch.pi * self.sigma2)
        )

        logp_epsilon = -0.5 * (
            epsilon ** 2 + math.log(2.0 * math.pi)
        ).sum(dim=1)

        return loglik + logp_epsilon

In [117]:
class NBase(nn.Module):
    def __init__(self, dim, init_loc=0.0, init_log_scale=-2.5):
        super().__init__()
        self.dim = dim
        self.loc = nn.Parameter(torch.full((dim,), float(init_loc)))
        self.raw_log_scale = nn.Parameter(torch.full((dim,), float(init_log_scale)))

    def log_scale(self):
        return torch.clamp(self.raw_log_scale, min=-5.0, max=2.0)

    def scale(self):
        return torch.exp(self.log_scale())

    def rsample(self, num_samples):
        eta = torch.randn(
            num_samples,
            self.dim,
            device=self.loc.device,
            dtype=self.loc.dtype,
        )
        z0 = self.loc.unsqueeze(0) + self.scale().unsqueeze(0) * eta
        return eta, z0

    def log_prob(self, z0):
        log_scale = self.log_scale().unsqueeze(0)
        var = torch.exp(2.0 * log_scale)
        return -0.5 * (
            ((z0 - self.loc.unsqueeze(0)) ** 2) / var
            + 2.0 * log_scale
            + math.log(2.0 * math.pi)
        ).sum(dim=1)

class FlowVI(nn.Module):
    def __init__(self, base_dist, transport, target_dist):
        super().__init__()
        self.base_dist = base_dist
        self.transport = transport          # posterior flow f_psi
        self.target_dist = target_dist      # contains model-side g_theta

    def sample_and_logq(self, num_samples):
        # eta: Gaussian reparameterization noise for the base sampler
        # z0 : base variational sample
        eta, z0 = self.base_dist.rsample(num_samples)

        # zK is epsilon in the advanced double-layer formulation
        zK, logdet = self.transport(z0, return_logdet=True)

        log_q0 = self.base_dist.log_prob(z0)
        log_qK = log_q0 - logdet

        return {
            "eta": eta,
            "z0": z0,
            "zK": zK,
            "epsilon": zK,
            "log_q0": log_q0,
            "log_qK": log_qK,
            "logdet": logdet,
        }

    def reverse_kld(self, num_samples=256, beta=1.0, return_parts=False):
        out = self.sample_and_logq(num_samples)

        log_p = self.target_dist.log_prob(out["zK"])
        loss = (out["log_qK"] - beta * log_p).mean()

        if return_parts:
            out["log_p"] = log_p
            out["loss"] = loss
            return out
        return loss

def build_flow_vi(
    target_dist,
    K=8,
    hidden_units=64,
    num_hidden_layers=2,
    learn_g_theta=True,
):
    dim = target_dist.dim

    # posterior base q0
    base = NBase(dim=dim, init_loc=0.0, init_log_scale=-2.5)

    # model-side transport g_theta
    if learn_g_theta and isinstance(target_dist.g_theta, IdentityTransport):
        target_dist.g_theta = TransportMap(
            dim=dim,
            K=K,
            hidden_units=hidden_units,
            num_hidden_layers=num_hidden_layers,
        )

    # posterior flow f_psi
    transport = TransportMap(
        dim=dim,
        K=K,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
    )

    model = FlowVI(base_dist=base, transport=transport, target_dist=target_dist)
    return model

In [122]:
def unique_parameters(*modules):
    params = []
    seen = set()
    for module in modules:
        if module is None:
            continue
        for p in module.parameters():
            if p.requires_grad and id(p) not in seen:
                seen.add(id(p))
                params.append(p)
    return params


def geometric_anneal(start, end, frac):
    if start <= 0 or end <= 0:
        raise ValueError("start and end must be positive for geometric annealing")
    return start * (end / start) ** frac


def train_flow_debug(
    model,
    target_dist,
    epochs=2000,
    num_samples=256,
    lr=3e-4,
    tau_start=1.0,
    tau_end=0.2,
    kl_anneal=True,
    grad_clip=5.0,
    print_every=100,
):
    optimizer = torch.optim.Adam(
        unique_parameters(model, target_dist),
        lr=lr
    )

    losses = []
    tau_hist = []
    grad_hist = []

    for it in range(epochs):
        optimizer.zero_grad()

        frac = it / max(epochs - 1, 1)

        tau_now = geometric_anneal(tau_start, tau_end, frac)
        target_dist.set_tau(tau_now)
        tau_hist.append(float(tau_now))

        anneal_beta = min(1.0, 0.01 + 0.99 * frac) if kl_anneal else 1.0

        loss = model.reverse_kld(
            num_samples=num_samples,
            beta=anneal_beta,
        )

        loss.backward()

        total_grad_sq = 0.0
        for p in unique_parameters(model, target_dist):
            if p.grad is not None:
                total_grad_sq += float((p.grad.detach() ** 2).sum().item())

        grad_norm = math.sqrt(total_grad_sq)
        grad_hist.append(grad_norm)

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(
                unique_parameters(model, target_dist),
                grad_clip
            )

        optimizer.step()
        losses.append(float(loss.item()))

        if (it + 1) % print_every == 0 or it == 0:
            print(
                f"epoch {it+1:5d} | "
                f"loss = {loss.item():.4f} | "
                f"tau = {tau_now:.4f} | "
                f"anneal_beta = {anneal_beta:.4f} | "
                f"grad_norm = {grad_norm:.4f}"
            )

    return losses, tau_hist, grad_hist

def train_flow_fixed_tau(
    model,
    target_dist,
    tau_fixed=0.5,
    epochs=1000,
    num_samples=128,
    lr=5e-5,
    grad_clip=3.0,
    print_every=100,
):
    return train_flow_debug(
        model=model,
        target_dist=target_dist,
        epochs=epochs,
        num_samples=num_samples,
        lr=lr,
        tau_start=tau_fixed,
        tau_end=tau_fixed,
        kl_anneal=False,
        grad_clip=grad_clip,
        print_every=print_every,
    )

In [104]:
@torch.no_grad()
def trace_eps(model, num_samples=4000, seed=123):
    torch.manual_seed(seed)

    # 1) base Gaussian reparameterization
    eta, z0 = model.base_dist.rsample(num_samples)

    # 2) posterior flow output: zK = epsilon
    zK, logdet = model.transport(z0, return_logdet=True)

    # 3) variational densities
    log_q0 = model.base_dist.log_prob(z0)
    log_qK = log_q0 - logdet

    # 4) target-side decode: epsilon -> xi = g_theta(epsilon) -> (s,u,t,m,beta)
    dec = model.target_dist.decode(zK)

    return {
        "eta": _to_cpu(eta),                              # Gaussian reparameterization noise
        "loc": _to_cpu(model.base_dist.loc),
        "scale": _to_cpu(model.base_dist.scale()),
        "log_scale": _to_cpu(model.base_dist.log_scale()),
        "z0": _to_cpu(z0),                                # base variational sample
        "zK": _to_cpu(zK),                                # posterior sample in epsilon-space
        "epsilon": _to_cpu(zK),                           # explicit alias
        "xi": _to_cpu(dec["xi"]),                         # primitive latent after g_theta
        "ldj_total": _to_cpu(logdet),
        "log_q0": _to_cpu(log_q0),
        "log_qK": _to_cpu(log_qK),
        "s": _to_cpu(dec["s"]),
        "u": _to_cpu(dec["u"]),
        "t": _to_cpu(dec["t"]),
        "m": _to_cpu(dec["m"]),
        "beta": _to_cpu(dec["beta"]),
    }

In [105]:
@torch.no_grad()
def posterior_summary(trace, beta_true, pip_cutoff=0.50):
    u = trace["u"]      # [S, p]
    t = trace["t"]      # [S, 1] or [S, p]

    beta_true_cpu = beta_true.detach().cpu()

    selected_each_draw = (u > t).float()   # broadcasting if t is [S,1]

    pip = selected_each_draw.mean(dim=0)

    truth = (beta_true_cpu.abs() > 1e-12).numpy().astype(int)
    pred = (pip.numpy() > pip_cutoff).astype(int)

    tp = int(((pred == 1) & (truth == 1)).sum())
    fp = int(((pred == 1) & (truth == 0)).sum())
    fn = int(((pred == 0) & (truth == 1)).sum())
    tn = int(((pred == 0) & (truth == 0)).sum())

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    df = pd.DataFrame({
        "j": np.arange(u.shape[1]),
        "beta_true": beta_true_cpu.numpy(),
        "pip": pip.numpy(),
        "truth": truth,
        "pred": pred,
    })

    metrics = {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "n_selected": int(pred.sum()),
        "selected_idx": np.where(pred == 1)[0].tolist(),
    }

    return df, metrics

In [118]:
def simfun1(
    n=180,
    p=100,
    seed=123,
    snr=3.0,
    true_prop = 0.1,
    device=None,
    dtype=torch.float32,
):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    X = rng.standard_normal((n, p), dtype=np.float32)
    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True) + 1e-8)

    n_active = int(p * true_prop)
    active_idx = np.sort(rng.choice(p, size=n_active, replace=False))

    beta_true = np.zeros(p, dtype=np.float32)
    magnitudes = rng.uniform(0.3, 2.0, size=n_active).astype(np.float32)
    signs = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=n_active)
    beta_true[active_idx] = signs * magnitudes

    signal = X @ beta_true
    signal_var = np.var(signal)
    sigma2 = signal_var / snr
    sigma = np.sqrt(sigma2)

    y = signal + sigma * rng.standard_normal(n, dtype=np.float32)
    y = y - y.mean()

    X_t = torch.tensor(X, dtype=dtype, device=device)
    y_t = torch.tensor(y, dtype=dtype, device=device)
    beta_true_t = torch.tensor(beta_true, dtype=dtype, device=device)

    info = {
        "n": n,
        "p": p,
        "n_active": n_active,
        "sigma2": float(sigma2),
        "sigma": float(sigma),
        "active_idx": active_idx,
        "beta_true": beta_true,
        "snr": snr,
    }
    return X_t, y_t, beta_true_t, info

In [123]:
def fixedtau1(
    seed=123,
    device=None,
    dtype=torch.float32,
    n=180,
    p=100,
    snr=3.0,
    true_prop=0.1,
    tau_fixed=0.5,
    K=4,
    hidden_units=64,
    num_hidden_layers=2,
    learn_g_theta=True,
    epochs=1000,
    num_samples=128,
    lr=5e-5,
    grad_clip=3.0,
    print_every=100,
    posterior_draws=4000,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X, y, beta_true, sim_info = simfun1(
        n=n,
        p=p,
        seed=seed,
        snr=snr,
        true_prop=true_prop,
        device=device,
        dtype=dtype,
    )

    print("===== Simulation info =====")
    print(sim_info)

    target_dist = Relaxedsas(
        X=X,
        y=y,
        sigma2=sim_info["sigma2"],
        tau=tau_fixed,
        g_theta=IdentityTransport(),
    ).to(device)

    model = build_flow_vi(
        target_dist=target_dist,
        K=K,
        hidden_units=hidden_units,
        num_hidden_layers=num_hidden_layers,
        learn_g_theta=learn_g_theta,
    ).to(device)

    losses, tau_hist, grad_hist = train_flow_fixed_tau(
        model=model,
        target_dist=target_dist,
        tau_fixed=tau_fixed,
        epochs=epochs,
        num_samples=num_samples,
        lr=lr,
        grad_clip=grad_clip,
        print_every=print_every,
    )

    trace = trace_eps(
        model,
        num_samples=posterior_draws,
        seed=seed + 999,
    )

    posterior_df, metrics = posterior_summary(
        trace=trace,
        beta_true=beta_true,
        pip_cutoff=0.50,
    )

    return {
        "beta_true": beta_true,
        "sim_info": sim_info,
        "model": model,
        "losses": losses,
        "tau_hist": tau_hist,
        "grad_hist": grad_hist,
        "trace": trace,
        "posterior_df": posterior_df,
        "metrics": metrics,
    }

In [110]:
out_hard = fixedtau1(
    seed=123,
    n=120,
    p=100,
    snr=2.5,
    true_prop=0.1,
    tau_fixed=0.05,
    K=8,
    hidden_units=64,
    num_hidden_layers=2,
    epochs=3000,
    num_samples=128,
    lr=5e-5,
    grad_clip=3.0,
    print_every=500,
    posterior_draws=4000,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 4.2100261688232425, 'sigma': 2.051834829810441, 'active_idx': array([ 0, 19, 24, 26, 32, 37, 58, 69, 74, 84], dtype=int64), 'beta_true': array([-1.2368672 ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        , -0.5543672 ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.77611005,
        0.        ,  0.72853726,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        , -1.4793472 ,  0.        ,  0.        ,
        0.        ,  0.        , -1.3470716 ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.      

In [114]:
trace_hard = trace_eps(
    out_hard["model"],
    num_samples=4000,
    seed=999,
)

posterior_df_hard, metrics_hard = posterior_summary(
    trace=trace_hard,
    beta_true=out_hard["beta_true"],
    pip_cutoff=0.50,
)

print(metrics_hard)
posterior_df_hard.query("pred == 1")

{'tp': 10, 'fp': 90, 'fn': 0, 'tn': 0, 'precision': 0.1, 'recall': 1.0, 'f1': 0.18181818181818182, 'n_selected': 100, 'selected_idx': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]}


,j,beta_true,pip,truth,pred
0,0,-1.236867,1.0,1,1
1,1,0.000000,1.0,0,1
2,2,0.000000,1.0,0,1
3,3,0.000000,1.0,0,1
4,4,0.000000,1.0,0,1
...,...,...,...,...,...
95,95,0.000000,1.0,0,1
96,96,0.000000,1.0,0,1
97,97,0.000000,1.0,0,1
98,98,0.000000,1.0,0,1


In [124]:
out_id = fixedtau1(
    seed=123,
    n=120,
    p=100,
    snr=2.5,
    true_prop=0.1,
    tau_fixed=0.05,
    K=8,
    hidden_units=64,
    num_hidden_layers=2,
    learn_g_theta=False,
    epochs=3000,
    num_samples=128,
    lr=5e-5,
    grad_clip=3.0,
    print_every=500,
    posterior_draws=4000,
)

===== Simulation info =====
{'n': 120, 'p': 100, 'n_active': 10, 'sigma2': 4.2100261688232425, 'sigma': 2.051834829810441, 'active_idx': array([ 0, 19, 24, 26, 32, 37, 58, 69, 74, 84], dtype=int64), 'beta_true': array([-1.2368672 ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        , -0.5543672 ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.77611005,
        0.        ,  0.72853726,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        , -1.4793472 ,  0.        ,  0.        ,
        0.        ,  0.        , -1.3470716 ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.      